# Protein Pretrain (Unified)

This notebook replaces the separate from-scratch, resume, and streaming notebooks.
It loads `train.yaml` to configure the full training flow.

Supports:
- **train_from_scratch**: Build tokenizer, model, and train from epoch 0.
- **resume**: Restore from a checkpoint and continue training.
- **auto**: Resume if checkpoint exists, otherwise train from scratch.

Data sources:
- Local `train.txt` or `train_part_*.txt` files
- MinIO/S3 streaming (downloads parts on demand)

After training, the model checkpoint is uploaded to **HuggingFace** (not S3).

> **Before running on Linux/local**: run this notebook from inside the repo, or set `MDNAC_REPO_DIR`.
> Put secrets in `.env` or export `HF_TOKEN`, `MICROBIAL_DATA_MINIO_ACCESS_KEY`, and `MICROBIAL_DATA_MINIO_SECRET_KEY`.

In [ ]:
# -- Linux/local setup ---------------------------------------------------------
import os
import subprocess
import sys
from pathlib import Path


def find_repo_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path

    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        repo_dir = Path(repo_dir_env).expanduser().resolve()
        if (repo_dir / "pyproject.toml").exists() and (repo_dir / "libs").is_dir():
            return repo_dir

    raise RuntimeError("Run this notebook from inside the MDNAC repo or set MDNAC_REPO_DIR=/path/to/MDNAC.")


def load_env_file(path: Path) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


REPO_DIR = find_repo_dir(Path.cwd())
os.chdir(REPO_DIR)
load_env_file(REPO_DIR / ".env")

if "MICROBIAL_DATA_MINIO_ACCESS_KEY" not in os.environ and "MINIO_ACCESS_KEY" in os.environ:
    os.environ["MICROBIAL_DATA_MINIO_ACCESS_KEY"] = os.environ["MINIO_ACCESS_KEY"]
if "MICROBIAL_DATA_MINIO_SECRET_KEY" not in os.environ and "MINIO_SECRET_KEY" in os.environ:
    os.environ["MICROBIAL_DATA_MINIO_SECRET_KEY"] = os.environ["MINIO_SECRET_KEY"]

os.environ.setdefault("MICROBIAL_DATA_STORAGE_MODE", "minio")
os.environ.setdefault("MICROBIAL_DATA_MINIO_ENDPOINT", "https://s3.phuongdong.cloud")
os.environ.setdefault("MICROBIAL_DATA_MINIO_BUCKET", "microbial-dna-compiler")
os.environ.setdefault("MICROBIAL_DATA_MINIO_REGION", "")
os.environ.setdefault("MICROBIAL_DATA_MINIO_SECURE", "true")
os.environ.setdefault("MICROBIAL_DATA_MINIO_PREFIX", "libs/data/models")

missing_env = [
    name
    for name in (
        "HF_TOKEN",
        "MICROBIAL_DATA_MINIO_ACCESS_KEY",
        "MICROBIAL_DATA_MINIO_SECRET_KEY",
    )
    if not os.environ.get(name)
]
if missing_env:
    raise RuntimeError(
        "Missing required environment variables: "
        + ", ".join(missing_env)
        + ". Export them in your shell or add them to .env."
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.core.pretrain.training_config import load_protein_training_config
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer
print(f"Setup complete: {REPO_DIR}")

In [ ]:
CONFIG_PATH = REPO_DIR / "train.16gb.yaml"

config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)

config_summary = {
    "mode": config["mode"],
    "optimizer_type": config["optimizer"]["type"],
    "device": config["runtime"]["device"],
    "multi_gpu_mode": config["runtime"]["multi_gpu_mode"],
    "mixed_precision": config["runtime"]["mixed_precision"],
    "checkpoint_dir": str(config["paths"]["checkpoint_dir"]),
    "resume_state_path": str(config["paths"]["resume_state_path"]),
    "minio_prefix": config["minio"]["train_parts_prefix_uri"] or None,
    "minio_uris_count": len(config["minio"]["train_part_uris"]),
    "train_part_cache_dir": str(config["paths"]["train_part_cache_dir"]),
    "num_epochs": config["training"]["num_epochs"],
    "max_steps": config["training"].get("max_steps"),
    "batch_size": config["data"]["batch_size"],
    "context_length": config["model"]["context_length"],
    "target_vram_gb": config["runtime"].get("target_vram_gb"),
}
config_summary

## VRAM Check

Before training, verify that the config fits within your GPU's VRAM budget.
This cell loads the real `tokenizer_map.json`, instantiates the model, and estimates memory usage.

In [ ]:
# ── VRAM Preflight Check ──────────────────────────────────────────────────────
from libs.data.training.tokenizer import SequenceTokenizer
from libs.core.pretrain.protein_lm.memory import (
    estimate_protein_pretrain_memory,
    _resolve_dtype_from_mixed_precision,
)
from libs.core.pretrain.protein_lm.support.backbone import (
    build_mdc_config_from_progen_config,
    build_progen_config,
)

# Load real tokenizer
tokenizer_map_path = config["paths"]["tokenizer_map_path"]
assert tokenizer_map_path.exists(), f"tokenizer_map.json not found: {tokenizer_map_path}"
tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
print(f"✅ Tokenizer loaded — vocab_size = {tokenizer.vocab_size}")

# Build model config with mixed precision dtype
mixed_precision = config["runtime"]["mixed_precision"]
resolved_dtype = _resolve_dtype_from_mixed_precision(mixed_precision)
model_cfg = config["model"]
progen_config = build_progen_config(
    model_cfg["progen_model_size"],
    vocab_size=tokenizer.vocab_size,
    context_length=model_cfg["context_length"],
    dtype=resolved_dtype,
)
overrides = model_cfg["progen_config_overrides"]
if overrides:
    progen_config = {**progen_config, **overrides}
model_config = build_mdc_config_from_progen_config(progen_config, dtype=resolved_dtype)

# Estimate VRAM
import torch
estimate = estimate_protein_pretrain_memory(
    model_config=model_config,
    tokenizer=tokenizer,
    batch_size=config["data"]["batch_size"],
    context_length=model_cfg["context_length"],
    optimizer_type=config["optimizer"]["type"],
    dtype=resolved_dtype,
    mixed_precision=mixed_precision,
)

runtime_cfg = config.get("runtime", {})
configured_target_vram_gb = runtime_cfg.get("target_vram_gb")

if configured_target_vram_gb is not None:
    target_budget = float(configured_target_vram_gb)
    budget_source = f"{CONFIG_PATH.name}: runtime.target_vram_gb"
elif torch.cuda.is_available():
    detected_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    target_budget = min(detected_vram_gb * 0.85, detected_vram_gb - 2.0)
    budget_source = "detected CUDA VRAM with safety margin"
else:
    raise RuntimeError("Set runtime.target_vram_gb in the training YAML before running VRAM preflight without CUDA.")

print(f"\n{'='*60}")
print(f"VRAM ESTIMATE (resolved_vocab_size={tokenizer.vocab_size})")
print(f"{'='*60}")
print(f"  Parameters:      {estimate['param_count']:,} ({estimate['param_memory_gb']:.3f} GB)")
print(f"  Gradients:       {estimate['gradient_memory_gb']:.3f} GB")
print(f"  Optimizer state: {estimate['optimizer_state_gb']:.3f} GB")
print(f"  Activations:     {estimate['activation_memory_gb']:.3f} GB")
print(f"  Logits:          {estimate['logits_memory_gb']:.3f} GB")
print(f"  TOTAL:           {estimate['total_estimate_gb']:.3f} GB")
print(f"  Budget:          {target_budget:.3f} GB ({budget_source})")
print(f"  Margin:          {target_budget - estimate['total_estimate_gb']:+.3f} GB")
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU total VRAM:  {total_vram_gb:.3f} GB")

fits = estimate["total_estimate_gb"] <= target_budget
if fits:
    print(f"\n  ✓ Estimated to fit within configured budget ({target_budget:.1f}GB)")
else:
    print(f"\n  ✗ EXCEEDS configured budget ({target_budget:.1f}GB)!")

if not torch.cuda.is_available():
    print("  ⚠ No CUDA — this is only an estimate. Actual peak may be higher.")

In [ ]:
USE_CONFIG = "current"

if USE_CONFIG == "16gb":
    CONFIG_PATH = REPO_DIR / "train.16gb.yaml"
    print("Using train.16gb.yaml")
elif USE_CONFIG == "current":
    # keep CONFIG_PATH as-is
    print(f"Using current config: {CONFIG_PATH}")
else:
    CONFIG_PATH = Path(USE_CONFIG)
    print(f"Using custom config: {CONFIG_PATH}")

# Reload config if changed
if USE_CONFIG != "current":
    config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)
    print(f"  batch_size={config['data']['batch_size']}, "
          f"context_length={config['model']['context_length']}, "
          f"grad_accum={config['training'].get('gradient_accumulation_steps', 1)}")

# Safety check: warn if estimated OOM
preflight = config.get("runtime", {}).get("preflight_vram_check", False) if isinstance(config.get("runtime"), dict) else False
if not fits and preflight:
    raise RuntimeError(
        f"⚠ Config estimated to use {estimate['total_estimate_gb']:.2f}GB "
        f"which exceeds target budget of {target_budget:.2f}GB. "
        f"Increase runtime.target_vram_gb in the YAML or reduce batch_size/context_length."
    )
elif not fits:
    print(f"⚠ WARNING: Config may exceed target VRAM budget ({target_budget:.2f}GB).")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = ProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
result = trainer.train()
result

In [ ]:
# ── Generate sample ───────────────────────────────────────────────────────────
from libs.core.pretrain.protein_lm.core import generate_protein_text

if trainer.is_main_process and trainer.model is not None:
    sample = generate_protein_text(
        trainer.model,
        trainer.tokenizer,
        prompt="<|protein|>",
        device=trainer.runtime.device,
        max_new_tokens=80,
        context_length=int(trainer.model_config.context_length),
    )
    print(sample)
else:
    print("Sample generation is emitted on rank 0 only.")

In [ ]:
# ── Upload model to HuggingFace ───────────────────────────────────────────────
from huggingface_hub import HfApi, login

HF_REPO_ID = "admincybers2/MDNAC-protein-pretrain"  # ← change repo name as needed

login(token=os.environ["HF_TOKEN"])
api = HfApi()

checkpoint_path = result.checkpoint_path
best_path = result.best_checkpoint_path

files_to_upload = []
if checkpoint_path and checkpoint_path.exists():
    files_to_upload.append((str(checkpoint_path), checkpoint_path.name))
if best_path and best_path.exists():
    files_to_upload.append((str(best_path), best_path.name))

# Upload tokenizer_map.json alongside
tokenizer_map = config["paths"]["tokenizer_map_path"]
if Path(tokenizer_map).exists():
    files_to_upload.append((str(tokenizer_map), "tokenizer_map.json"))

# Upload resume_state.json
resume_state = config["paths"]["resume_state_path"]
if Path(resume_state).exists():
    files_to_upload.append((str(resume_state), "resume_state.json"))

for local_path, repo_filename in files_to_upload:
    print(f"Uploading {repo_filename} ...")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_filename,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"✅ Uploaded {len(files_to_upload)} file(s) to https://huggingface.co/{HF_REPO_ID}")